# Review a candidate adapter

Run this notebook only after `01_reproduce_mft_gemma3.ipynb` has produced a completed candidate adapter and FT face-sanity bundle. This notebook does **not** fine-tune. It (1) evaluates the base model on the same frozen face role, (2) creates a blinded candidate-adapter review CSV, (3) offers a protected candidate-adapter upload, and (4) can run an explicitly labelled plumbing extraction.

Face-domain sanity does **not** reproduce OOD EM. Primary RQ1 is disabled until a sealed paper-comparable OOD baseline (150 broad text prompts + 250 LLaVA/MSCOCO VQA pairs) is reviewed across seeds 42, 43, and 44. Do not begin RQ2 or RQ3 here.

## 1. Required runtime

Use Colab A100. This notebook loads the base and adapter sequentially, so it does not need two models in GPU memory at once.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'A100' in GPU_NAME, 'Use an A100 runtime for this candidate review.'
assert torch.cuda.is_bf16_supported(), 'bf16 support is required.'


## 2. Mount Drive and use the current repository

Keep the same `DRIVE_PROJECT` and training seed used for the completed FT. Every adapter review reuses the immutable data-selection seed 42 split. The notebook fails rather than silently using a stale or locally edited clone.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

SEED = 42  # adapter training seed
DATA_SELECTION_SEED = 42  # fixed shared faces split
HUB_NAMESPACE = 'rlogger'  # Change only if your Hugging Face namespace differs.
DRIVE_PROJECT = Path('/content/drive/MyDrive/em-displacement-vlm')
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)

for subdir in ('data', 'checkpoints', 'results', 'runs', 'wandb'):
    (DRIVE_PROJECT / subdir).mkdir(parents=True, exist_ok=True)
os.environ['EM_DATA_DIR'] = str(DRIVE_PROJECT / 'data')
os.environ['EM_CHECKPOINT_DIR'] = str(DRIVE_PROJECT / 'checkpoints')
os.environ['EM_RESULTS_DIR'] = str(DRIVE_PROJECT / 'results')
os.environ['HF_HOME'] = '/content/hf-cache'

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    raise SystemExit('Secret not set: HF_TOKEN. Add it in Colab before model access or Hub upload.')
os.environ['HF_TOKEN'] = HF_TOKEN
SPLIT_ROOT = DRIVE_PROJECT / 'data' / 'splits' / f'seed{DATA_SELECTION_SEED}'
ADAPTER_DIR = DRIVE_PROJECT / 'checkpoints' / f'FT_R32_gemma3_faces_seed{SEED}'
HUB_REPO = f'{HUB_NAMESPACE}/FT_R32_gemma3_faces_seed{SEED}'
assert (SPLIT_ROOT / 'manifest.json').is_file(), f'Missing frozen split: {SPLIT_ROOT}'
split_manifest = json.loads((SPLIT_ROOT / 'manifest.json').read_text())
assert split_manifest.get('seed') == DATA_SELECTION_SEED, split_manifest
assert (ADAPTER_DIR / 'adapter_config.json').is_file(), f'Missing completed adapter: {ADAPTER_DIR}'

REPO_URL = 'https://github.com/rlogger/em-displacement-vlm.git'
REPO_DIR = Path('/content/em-displacement-vlm')
if REPO_DIR.exists():
    assert (REPO_DIR / '.git').is_dir(), f'{REPO_DIR} exists but is not a clone; restart the runtime.'
    assert not subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip(), 'Clone is dirty; restart the runtime.'
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main'])
else:
    subprocess.check_call(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, str(REPO_DIR)])
%cd {REPO_DIR}
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


In [ ]:
import json
from importlib.metadata import PackageNotFoundError, version

constraints_path = REPO_DIR / 'constraints' / 'colab.txt'
assert constraints_path.is_file(), f'Missing constraints file: {constraints_path}'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--constraint', str(constraints_path), 'unsloth', 'datasets>=2.19',
    'huggingface-hub>=0.23', 'safetensors>=0.4', 'pyyaml>=6.0', 'peft',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'])
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
fresh_runtime_output = subprocess.check_output([
    sys.executable, '-c',
    "import json, torch, unsloth; print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'unsloth': 'OK'}))",
], text=True)
fresh_runtime = json.loads(fresh_runtime_output.strip().splitlines()[-1])
print(fresh_runtime)

def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None

runtime_manifest = {
    'seed': SEED, 'data_selection_seed': DATA_SELECTION_SEED, 'git_commit': REPO_COMMIT, 'python': sys.version,
    'gpu': GPU_NAME, 'torch': fresh_runtime['torch'], 'cuda': fresh_runtime['cuda'],
    'packages': {name: _package_version(name) for name in (
        'unsloth', 'transformers', 'trl', 'peft', 'datasets', 'safetensors'
    )},
    'pip_freeze': subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).splitlines(),
}
RUNTIME_MANIFEST = DRIVE_PROJECT / 'runs' / f'environment_candidate_review_seed{SEED}.json'
serialized_runtime_manifest = json.dumps(runtime_manifest, indent=2, sort_keys=True) + '\n'
if RUNTIME_MANIFEST.exists() and RUNTIME_MANIFEST.read_text() != serialized_runtime_manifest:
    raise RuntimeError(f'Runtime differs from frozen manifest: {RUNTIME_MANIFEST}')
if not RUNTIME_MANIFEST.exists():
    RUNTIME_MANIFEST.write_text(serialized_runtime_manifest)
print('Runtime manifest:', RUNTIME_MANIFEST)


## 3. Evaluate the base model on the exact same face-sanity role

The existing FT face-sanity bundle alone cannot establish a model change. This generates the matched base bundle. It does not modify the adapter or frozen split, and it remains a candidate-adapter check rather than OOD EM evidence.

In [ ]:
import yaml

BASE_MODEL_ID = 'unsloth/gemma-3-4b-it'
BASE_MODEL_REVISION = 'bf46152c47f5dd20b896357cb51abc4c03b8ee8c'
base_cfg = yaml.safe_load(Path('configs/sanity_em.yaml').read_text())
base_cfg.update({
    'model_id': BASE_MODEL_ID,
    'base_model_id': BASE_MODEL_ID,
    'base_model_revision': BASE_MODEL_REVISION,
    'seed': SEED,
    'data_selection_seed': DATA_SELECTION_SEED,
    'run_name': f'verify_base_gemma3_seed{SEED}_bf16',
    'split_root': str(SPLIT_ROOT),
    'load_in_4bit': False,
    'use_wandb': False,
})
BASE_SANITY_CONFIG = DRIVE_PROJECT / 'runs' / f'verify_base_gemma3_seed{SEED}_bf16.yaml'
rendered = yaml.safe_dump(base_cfg, sort_keys=False)
if BASE_SANITY_CONFIG.exists() and BASE_SANITY_CONFIG.read_text() != rendered:
    raise RuntimeError(f'Existing base config differs: {BASE_SANITY_CONFIG}')
BASE_SANITY_CONFIG.write_text(rendered)
BASE_BUNDLE = DRIVE_PROJECT / 'results' / f"sanity_checks_{base_cfg['run_name']}.json"
print('Base config:', BASE_SANITY_CONFIG)
print('Base bundle:', BASE_BUNDLE)


In [ ]:
if not BASE_BUNDLE.exists():
    subprocess.check_call([sys.executable, 'scripts/sanity_check_em.py', '--config', str(BASE_SANITY_CONFIG)])
else:
    print('Reusing existing base sanity bundle:', BASE_BUNDLE)


## 4. Create and complete the blinded candidate-adapter review

Label the response, not the face or prompt. Use `docs/BEHAVIORAL_REVIEW.md` for the rubric. The review is an outcome measurement only; it is never used as training data or to choose activations after seeing the result. It does not establish the separate OOD EM reproduction gate.

In [ ]:
FT_BUNDLE = DRIVE_PROJECT / 'results' / f"sanity_checks_verify_mft_gemma3_seed{SEED}_bf16.json"
assert FT_BUNDLE.is_file(), f'Missing FT sanity bundle: {FT_BUNDLE}'
assert BASE_BUNDLE.is_file(), f'Missing base sanity bundle: {BASE_BUNDLE}'
for bundle in (BASE_BUNDLE, FT_BUNDLE):
    assert bundle.with_suffix('.meta.json').is_file(), f'Missing provenance sidecar: {bundle}'
REVIEW_SHEET = DRIVE_PROJECT / 'results' / f'review_seed{SEED}.csv'
REVIEW_MAPPING = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_mapping.json'
if REVIEW_SHEET.exists() != REVIEW_MAPPING.exists():
    raise RuntimeError(
        'Partial review package found. Preserve it for audit, then use fresh review-sheet and mapping paths; '
        'neither file is sufficient alone.'
    )
if not REVIEW_SHEET.exists():
    subprocess.check_call([
        sys.executable, 'scripts/make_annotation_sheet.py',
        '--bundle', 'base', str(BASE_BUNDLE),
        '--bundle', 'ft', str(FT_BUNDLE),
        '--out', str(REVIEW_SHEET),
        '--mapping-out', str(REVIEW_MAPPING),
        '--seed', str(SEED),
    ])
print('Complete this CSV before unblinding:', REVIEW_SHEET)
print('Keep hidden until review is complete:', REVIEW_MAPPING)


Download or open the CSV in a spreadsheet, complete every label/confidence field, then save it back to Drive as `review_seed42_completed.csv`. Use a second reviewer on a stratified subset if possible. Only after unblinding may you set the explicit candidate-adapter decision below. A face-sanity decision never substitutes for the separate OOD paper-comparable gate.

In [ ]:
COMPLETED_REVIEW = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_completed.csv'
REVIEW_SUMMARY = DRIVE_PROJECT / 'results' / f'review_seed{SEED}_summary.json'

# Manual decision: keep the default until every response has been reviewed.
CANDIDATE_FACE_SANITY_GATE = 'undecided'  # allowed: pass, fail, undecided
CANDIDATE_DECISION_RATIONALE = ''  # Required for pass or fail.
REVIEWER_ID = ''  # Optional pseudonymous reviewer ID.
CANDIDATE_REVIEW_CONFIRMATION = ''
expected_confirmation = f'reviewed candidate face sanity seed {SEED}'
assert CANDIDATE_FACE_SANITY_GATE in {'pass', 'fail', 'undecided'}
if CANDIDATE_REVIEW_CONFIRMATION != expected_confirmation:
    raise RuntimeError(
        'Complete the blinded CSV, unblind it, record pass/fail/undecided, then set '
        f'CANDIDATE_REVIEW_CONFIRMATION to {expected_confirmation!r}. '
        'This is a candidate-adapter face-sanity gate, not an OOD EM reproduction claim.'
    )
assert COMPLETED_REVIEW.is_file(), f'Complete and upload the CSV first: {COMPLETED_REVIEW}'
subprocess.check_call([
    sys.executable, 'scripts/summarize_annotation_sheet.py',
    '--input', str(COMPLETED_REVIEW),
    '--mapping', str(REVIEW_MAPPING),
    '--out', str(REVIEW_SUMMARY),
    '--behavioral-gate', CANDIDATE_FACE_SANITY_GATE,
    '--decision-rationale', CANDIDATE_DECISION_RATIONALE,
    '--reviewer-id', REVIEWER_ID,
])
review_summary = json.loads(REVIEW_SUMMARY.read_text())
assert review_summary.get('behavioral_gate') == CANDIDATE_FACE_SANITY_GATE, review_summary
if CANDIDATE_FACE_SANITY_GATE != 'pass':
    print('Candidate did not pass. Preserve the record; do not publish or run extraction.')
print('Candidate-adapter face-sanity review saved:', REVIEW_SUMMARY)


## 5. Optional private candidate-adapter upload

This is disabled by default. The protected command requires the completed review summary; publishing retains a **candidate adapter**, not a reproduced-EM model. Keep recovery checkpoints private on Drive.

In [ ]:
PUBLISH_CANDIDATE_ADAPTER = False
if PUBLISH_CANDIDATE_ADAPTER:
    assert REVIEW_SUMMARY.is_file(), 'Complete the candidate review first.'
    subprocess.check_call([
        sys.executable, 'scripts/push_adapter.py',
        '--adapter-dir', str(ADAPTER_DIR),
        '--repo-id', HUB_REPO,
        '--review-summary', str(REVIEW_SUMMARY),
        '--evidence-tier', 'candidate',
    ])
else:
    print('Candidate upload is disabled. Set PUBLISH_CANDIDATE_ADAPTER=True only after completed review.')


In [ ]:
# Optional plumbing extraction — this has no RQ1 claim.
RUN_SEED42_PLUMBING_PILOT = False
if RUN_SEED42_PLUMBING_PILOT:
    assert REVIEW_SUMMARY.is_file(), 'Run and record the candidate-adapter review first.'
    assert json.loads(REVIEW_SUMMARY.read_text()).get('behavioral_gate') == 'pass', (
        'Plumbing extraction requires a passed candidate face-sanity review.'
    )
    rq1_cfg = yaml.safe_load(Path('configs/extract_rq1.yaml').read_text())
    RQ1_DIR = DRIVE_PROJECT / 'results' / f'rq1_plumbing_seed{SEED}'
    rq1_cfg.update({
        'analysis_tier': 'plumbing_pilot',
        'run_name': f'extract_rq1_plumbing_seed{SEED}',
        'seed': SEED,
        'data_selection_seed': DATA_SELECTION_SEED,
        'ft_adapter': str(ADAPTER_DIR),
        'split_root': str(SPLIT_ROOT),
        'output_dir': str(RQ1_DIR),
        'review_summary': str(REVIEW_SUMMARY),
    })
    RQ1_CONFIG = DRIVE_PROJECT / 'runs' / f'extract_rq1_plumbing_seed{SEED}.yaml'
    rendered = yaml.safe_dump(rq1_cfg, sort_keys=False)
    if RQ1_CONFIG.exists() and RQ1_CONFIG.read_text() != rendered:
        raise RuntimeError(f'Existing plumbing config differs: {RQ1_CONFIG}')
    RQ1_CONFIG.write_text(rendered)
    subprocess.check_call([sys.executable, 'scripts/extract_rq1.py', '--config', str(RQ1_CONFIG)])
    RQ1_BUNDLE = RQ1_DIR / 'rq1_geometry.json'
    assert RQ1_BUNDLE.is_file(), RQ1_BUNDLE
    print(RQ1_BUNDLE.read_text())
else:
    print('Plumbing extraction is disabled. This notebook has not produced a primary RQ1 result.')


## 6. Primary RQ1 preflight — intentionally disabled

Primary RQ1 requires the passed `ood_three_seed_gate.json` created by notebook 03. That gate cryptographically binds all three calibrated seed reviews, pair fingerprints, judge summaries, exact adapter fingerprints, reproduction manifests, and frozen splits. It also needs at least 50 unique matched prompt/image pairs plus sealed primary and control prompt manifests with approved review metadata (`review_status`, manifest hash, reviewer, date, selection policy). Face-sanity evidence cannot satisfy this gate. Continue with notebook 03, then notebook 04; the shared-residual analysis remains an extension to the paper's final-token/SVD geometry.

In [ ]:
PRIMARY_RQ1_READY = False
if PRIMARY_RQ1_READY:
    raise RuntimeError(
        'Do not enable this placeholder from face-sanity evidence. First create the sealed OOD '
        'paper-comparable baseline and manifests described above; the extractor will validate their provenance.'
    )
print(
    'Primary RQ1 remains disabled: no reviewed OOD baseline/provenance sidecar is present in this notebook. '
    'This is the expected state before the OOD evaluation package exists.'
)
